# Factor Engineering Template

Template notebook for building and screening equity factors on a point-in-time S&P 500 universe.

**Workflow:** load panel → engineer factors (long format) → optional save → Alphalens tear sheet.

**Backtest integrity:** A signal at date `t` must use only information available at or before `t`. Lag rolling features one bar (`.shift(1)` within each ticker) so values at `t` reflect completed bars through `t-1`.

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from data.ingestion.equity_fetcher import fetch_top_n_equities

## Data loading

Fetch daily OHLCV for the top 100 S&P 500 names by trailing dollar volume as of the start date.

- **Universe:** point-in-time S&P 500 membership (no survivorship bias).
- **Ranking:** 20 business days of dollar volume ending on or before `start_date`.
- **Source:** Yahoo Finance via `yfinance` (first run may take several minutes).
- Set `cache_dir` to `DEFAULT_CACHE_DIR` from `equity_fetcher` to speed up reruns.

In [ ]:
panel = fetch_top_n_equities(
    n=100,
    start_date="2020-01-02",
    end_date=None,
    lookback_days=20,
    cache_dir=None,
)

print(f"shape: {panel.shape}")
print(f"tickers: {panel['ticker'].nunique()}")
print(f"date range: {panel['date'].min().date()} → {panel['date'].max().date()}")
panel.head()

## Factor engineering

Add each factor as a new column on `panel`. **Stay in long format** — do not pivot to wide here.

Duplicate the RSI pattern below for each new factor:
1. Compute per ticker with `groupby("ticker").transform(...)`
2. Lag one bar with `.shift(1)` so the value at `t` uses data through `t-1`
3. Assign to `panel["your_factor_name"]`

### RSI (14)
Mean-reversion oscillator; high RSI = overbought.

### Other factors (add alongside)
- **MACD** — trend/momentum crossover
- **12-1 momentum** — return over 12 months skipping the most recent month
- **20-day volatility** — rolling std of daily returns
- **Cross-sectional z-score** — rank or z-score returns within each date

In [ ]:
def compute_rsi(close: pd.Series, period: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = (-delta).clip(lower=0)
    avg_gain = gain.ewm(alpha=1 / period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / period, min_periods=period, adjust=False).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))


panel["rsi_14"] = panel.groupby("ticker", sort=False)["close"].transform(
    lambda s: compute_rsi(s, period=14).shift(1)
)

panel[["date", "ticker", "close", "rsi_14"]].dropna().head()

## Alphalens tear sheet

### Why wide format — and only here

Alphalens' API requires **wide-format `prices`** (dates × tickers) and a factor `Series` with MultiIndex `(date, ticker)`. This is a library constraint from the Quantopian stack, not a requirement of your project.

The pivot to wide happens **only in the next cell**, immediately before calling Alphalens. All prior work remains long so you can save or reuse `panel` without Alphalens-specific structure.

### Alphalens does not need a manually shifted forward return

You do **not** need to create `close.shift(-1)` or `pct_change().shift(-1)` for Alphalens. Pass `prices` and `periods=(1, 5, 10)` to `get_clean_factor_and_forward_returns` — it computes forward returns internally from the price matrix.

### What to look for in the tear sheet

- **Mean returns by quantile** — monotonic spread (Q5 > Q1) suggests edge; flat = none.
- **Cumulative returns by quantile** — stable separation over time.
- **IC (information coefficient)** — daily rank correlation between factor and forward return.
- **Turnover** — how much the top quantile changes (cost sensitivity).

RSI is often **contrarian** (fade high RSI). Negative IC can still be tradable — invert the factor or go short high RSI. Log hypotheses in `02_research/hypothesis_log.md`.

### Other Alphalens tools

| Function | Use when |
|----------|----------|
| `al.tears.create_returns_tear_sheet` | Returns / quantile spread only |
| `al.tears.create_information_tear_sheet` | IC, IC decay |
| `al.tears.create_turnover_tear_sheet` | Capacity / cost sensitivity |
| `al.plotting.plot_ic_ts` / `plot_ic_hist` | IC stability visuals |

In [ ]:
import alphalens as al

# Last prep step — wide conversion for Alphalens only
prices = panel.pivot(index="date", columns="ticker", values="close")
factor = panel.set_index(["date", "ticker"])["rsi_14"]
factor = factor.dropna()

factor.index = factor.index.set_levels(
    pd.to_datetime(factor.index.levels[0]), level=0
)
prices.index = pd.to_datetime(prices.index)

factor_data = al.utils.get_clean_factor_and_forward_returns(
    factor=factor,
    prices=prices,
    quantiles=5,
    periods=(1, 5, 10),
    max_loss=0.35,
)

al.tears.create_full_tear_sheet(factor_data, long_short=True)